In [1]:
# ============================================================
# Notebook    : 14b_case_c_mlp.ipynb
# Description : Case C — MLP contrast architecture (§8.5).
#               Builds C1-MLP and C2-MLP using the EXACT SAME
#               train/val/test split as 08a/08b/13.
#
#               Purpose: confirm that BonusMalus's significant
#               effect (§4.8, LightGBM +0.0289, p<0.001) is a
#               property of the DATA, not an artifact of the
#               tree-based architecture.
# ============================================================


# ============================================================
# 0. Install dependencies (run once)
# ============================================================
# pip install torch scikit-learn pandas numpy


# ============================================================
# 1. Common imports
# ============================================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cpu")


# ============================================================
# 2. Load and split — IDENTICAL to notebook 13's Case C block
# ============================================================
df_c = pd.read_csv("data/fremtpl2_preprocessed.csv")

NUMERIC_COLS_C1 = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density"]
CAT_COLS_C = ["VehBrand", "VehGas", "Area", "Region"]
FEATURE_COLS_C1 = NUMERIC_COLS_C1 + CAT_COLS_C
FEATURE_COLS_C2 = ["Exposure", "VehPower", "VehAge", "DrivAge", "Density", "BonusMalus"] + CAT_COLS_C

X_c_full = df_c[FEATURE_COLS_C2].copy()
y_c_full = df_c["Label"].copy()

X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_c_full, y_c_full, test_size=0.30, random_state=SEED, stratify=y_c_full
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.50, random_state=SEED, stratify=y_temp_c
)

print(f"[CHECK 1] Split sizes — train: {len(X_train_c):,}, val: {len(X_val_c):,}, test: {len(X_test_c):,}")


# ============================================================
# 3. Preprocessing helper — same pattern as 14a
#    NOTE: Region has 21 categories, VehBrand has 11 — one-hot
#    will add ~35+ columns just from these two. Fine for MLP
#    at this scale (474K training rows).
# ============================================================
def build_mlp_features(X_train, X_val, X_test, numeric_cols, cat_cols):
    X_train_oh = pd.get_dummies(X_train[cat_cols], columns=cat_cols)
    X_val_oh   = pd.get_dummies(X_val[cat_cols],   columns=cat_cols)
    X_test_oh  = pd.get_dummies(X_test[cat_cols],  columns=cat_cols)

    X_val_oh  = X_val_oh.reindex(columns=X_train_oh.columns, fill_value=0)
    X_test_oh = X_test_oh.reindex(columns=X_train_oh.columns, fill_value=0)

    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train[numeric_cols])
    X_val_num   = scaler.transform(X_val[numeric_cols])
    X_test_num  = scaler.transform(X_test[numeric_cols])

    X_train_final = np.concatenate([X_train_num, X_train_oh.values.astype(np.float32)], axis=1)
    X_val_final   = np.concatenate([X_val_num,   X_val_oh.values.astype(np.float32)],   axis=1)
    X_test_final  = np.concatenate([X_test_num,  X_test_oh.values.astype(np.float32)],  axis=1)

    return X_train_final, X_val_final, X_test_final


# ============================================================
# 4. MLP model and training function — IDENTICAL to 14a
#    (same architecture, so any AUC difference vs 14a's B1/B2
#    reflects the DATA, not a differently-tuned model)
# ============================================================
class MLP(nn.Module):
    def __init__(self, input_dim, hidden1=64, hidden2=32, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden1, hidden2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_train, y_train, X_val, y_val, n_epochs=30, lr=1e-3, patience=5, batch_size=512):
    # NOTE: batch_size raised to 512 and n_epochs/patience lowered
    # vs 14a — Case C has 474K training rows (vs Case B's 7K), so
    # smaller batches / more epochs would be needlessly slow for
    # a contrast-architecture check, not the primary result
    input_dim = X_train.shape[1]
    model = MLP(input_dim).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())

    pos_rate = y_train.mean()
    pos_weight = torch.tensor((1 - pos_rate) / pos_rate)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train.values, dtype=torch.float32)
    X_val_t   = torch.tensor(X_val, dtype=torch.float32)
    y_val_t   = torch.tensor(y_val.values, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)

    best_val_auc = -1
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits).numpy()
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(f"  epoch {epoch:3d} | val_auc: {val_auc:.4f} | best: {best_val_auc:.4f}")

        if epochs_no_improve >= patience:
            print(f"  early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, n_params, best_val_auc


# ============================================================
# 5. Train C1-MLP (static, no BonusMalus)
# ============================================================
print("\n" + "="*60)
print("C1-MLP — Static only")
print("="*60)

X_train_c1, X_val_c1, X_test_c1 = build_mlp_features(
    X_train_c, X_val_c, X_test_c, NUMERIC_COLS_C1, CAT_COLS_C
)
print(f"[CHECK C1] Feature dim after encoding: {X_train_c1.shape[1]}")

model_c1_mlp, n_params_c1, best_val_auc_c1 = train_mlp(
    X_train_c1, y_train_c, X_val_c1, y_val_c
)
print(f"[CHECK C1] Params: {n_params_c1:,}")

model_c1_mlp.eval()
with torch.no_grad():
    prob_c1_mlp = torch.sigmoid(model_c1_mlp(torch.tensor(X_test_c1, dtype=torch.float32))).numpy()

auc_c1_mlp = roc_auc_score(y_test_c.values, prob_c1_mlp)
print(f"[CHECK C1] Test AUC (C1-MLP): {auc_c1_mlp:.4f}")


# ============================================================
# 6. Train C2-MLP (static + BonusMalus)
# ============================================================
print("\n" + "="*60)
print("C2-MLP — Static + BonusMalus")
print("="*60)

X_train_c2, X_val_c2, X_test_c2 = build_mlp_features(
    X_train_c, X_val_c, X_test_c, NUMERIC_COLS_C1 + ["BonusMalus"], CAT_COLS_C
)
print(f"[CHECK C2] Feature dim after encoding: {X_train_c2.shape[1]}")

model_c2_mlp, n_params_c2, best_val_auc_c2 = train_mlp(
    X_train_c2, y_train_c, X_val_c2, y_val_c
)
print(f"[CHECK C2] Params: {n_params_c2:,}")

model_c2_mlp.eval()
with torch.no_grad():
    prob_c2_mlp = torch.sigmoid(model_c2_mlp(torch.tensor(X_test_c2, dtype=torch.float32))).numpy()

auc_c2_mlp = roc_auc_score(y_test_c.values, prob_c2_mlp)
print(f"[CHECK C2] Test AUC (C2-MLP): {auc_c2_mlp:.4f}")


# ============================================================
# 7. Paired bootstrap significance
# ============================================================
def paired_bootstrap_auc_diff(y_true, prob_1, prob_2, n_boot=1000, seed=42, alpha=0.05):
    y_true = np.asarray(y_true)
    n = len(y_true)
    point_diff = roc_auc_score(y_true, prob_2) - roc_auc_score(y_true, prob_1)
    rng = np.random.RandomState(seed)
    diffs = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.randint(0, n, size=n)
        yt_b = y_true[idx]
        if len(np.unique(yt_b)) < 2:
            diffs[b] = np.nan
            continue
        diffs[b] = roc_auc_score(yt_b, prob_2[idx]) - roc_auc_score(yt_b, prob_1[idx])
    diffs = diffs[~np.isnan(diffs)]
    ci_low, ci_high = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    p_approx = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return point_diff, ci_low, ci_high, not (ci_low <= 0 <= ci_high), p_approx

diff_c, ci_low_c, ci_high_c, sig_c, p_c = paired_bootstrap_auc_diff(
    y_test_c.values, prob_c1_mlp, prob_c2_mlp
)
print(f"\n[SIGNIFICANCE] C1-MLP vs C2-MLP: diff={diff_c:+.4f}, "
      f"95% CI=[{ci_low_c:+.4f}, {ci_high_c:+.4f}], "
      f"significant={sig_c}, p_approx={p_c:.4f}")


# ============================================================
# 8. Save
# ============================================================
torch.save(model_c1_mlp.state_dict(), "data/sequences/case_c_mlp_static_model.pt")
torch.save(model_c2_mlp.state_dict(), "data/sequences/case_c_mlp_bonusmalus_model.pt")
np.savez("data/sequences/case_c_mlp_test_predictions.npz",
         labels=y_test_c.values, prob_c1_mlp=prob_c1_mlp, prob_c2_mlp=prob_c2_mlp)
print("\nSaved MLP models and predictions.")


# ============================================================
# 9. Comparison table
# ============================================================
print("\n===== Case C: Architecture Comparison Summary =====")
comparison_c = pd.DataFrame([
    {"Feature_Set": "Static only", "LightGBM_AUC": 0.6788, "MLP_AUC": round(auc_c1_mlp, 4)},
    {"Feature_Set": "Static+BonusMalus", "LightGBM_AUC": 0.7077, "MLP_AUC": round(auc_c2_mlp, 4)},
])
print(comparison_c.to_string(index=False))
print(f"\nLightGBM improvement (C1->C2): +0.0289 (significant, p<0.001, from notebook 13)")
print(f"MLP improvement (C1-MLP->C2-MLP): {diff_c:+.4f} (significant={sig_c}, p_approx={p_c:.4f})")

comparison_c.to_csv("paper_assets/tables/Table5_CaseC_Architecture_Comparison.csv", index=False)
print("\nSaved: paper_assets/tables/Table5_CaseC_Architecture_Comparison.csv")

[CHECK 1] Split sizes — train: 474,609, val: 101,702, test: 101,702

C1-MLP — Static only
[CHECK C1] Feature dim after encoding: 45
  epoch   1 | val_auc: 0.6322 | best: 0.6322
  epoch   2 | val_auc: 0.6508 | best: 0.6508
  epoch   3 | val_auc: 0.6540 | best: 0.6540
  epoch   4 | val_auc: 0.6599 | best: 0.6599
  epoch   5 | val_auc: 0.6662 | best: 0.6662
  epoch   6 | val_auc: 0.6672 | best: 0.6672
  epoch   7 | val_auc: 0.6690 | best: 0.6690
  epoch   8 | val_auc: 0.6687 | best: 0.6690
  epoch   9 | val_auc: 0.6690 | best: 0.6690
  epoch  10 | val_auc: 0.6673 | best: 0.6690
  epoch  11 | val_auc: 0.6687 | best: 0.6690
  epoch  12 | val_auc: 0.6698 | best: 0.6698
  epoch  13 | val_auc: 0.6695 | best: 0.6698
  epoch  14 | val_auc: 0.6686 | best: 0.6698
  epoch  15 | val_auc: 0.6687 | best: 0.6698
  epoch  16 | val_auc: 0.6681 | best: 0.6698
  epoch  17 | val_auc: 0.6704 | best: 0.6704
  epoch  18 | val_auc: 0.6697 | best: 0.6704
  epoch  19 | val_auc: 0.6676 | best: 0.6704
  epoch  20 |